# **HANDS - ON: FUNDAMENTOS DE LLMS CON MODELOS LLAMA**


Una vez vista la masterclass ***Fundamentos de LLMs y Arquitectura de Llama***, se proporciona el siguiente ***Colab*** para construir, en vivo, la primera llamada a Llama y medir su comportamiento.

Usamos **Ollama** para ejecutar Llama directamente en la GPU gratuita de Colab, sin necesidad de API key, cuentas externas ni infraestructura propia.

De igual manera, se proporciona la **solución** de este notebook a través del siguiente [enlace](https://colab.research.google.com/drive/1bjMHw7fmMxo9dyQp140e9mqotKRkfKBg?usp=sharing).

## **CONFIGURACIÓN DEL ENTORNO**

Antes de llamar a Llama necesitamos activar la GPU gratuita de Colab, instalar **Ollama** (el programa que descarga y ejecuta los modelos) y descargar los modelos que vamos a usar. La librería `ollama` es el cliente oficial en Python para hablar con ese servidor.

### **ACTIVAR LA GPU**

En el menú de Colab ve a **Entorno de ejecución → Cambiar tipo de entorno de ejecución** y selecciona **T4 GPU**. Luego ejecuta la celda para confirmar que Colab te asignó la GPU.

In [ ]:
# Verificar que Colab asignó una GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

name, memory.total [MiB]
Tesla T4, 15360 MiB


### **INSTALAR OLLAMA**

A diferencia de una API en la nube, aquí **el modelo corre dentro de tu propia sesión de Colab**: no hay *API key* ni *Secrets*. Ollama se instala en la máquina virtual, se arranca como un servidor local (`localhost:11434`) y desde Python le enviamos los prompts.

**IMPORTANTE:** la sesión de Colab es temporal. Si se reinicia o se desconecta, hay que volver a ejecutar las celdas de instalación, servidor y descarga de modelos.

In [ ]:
# Instalar Ollama en la máquina virtual de Colab (tarda ~1 minuto)
!sudo apt-get -qq install -y zstd > /dev/null 2>&1

!curl -fsSL https://ollama.com/install.sh | sh > /dev/null 2>&1

print("Ollama instalado.")


Ollama instalado.


In [ ]:
# Arrancar el servidor de Ollama en segundo plano y esperar a que responda
import subprocess

import time

import requests

servidor = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for _ in range(30):

    try:

        requests.get("http://localhost:11434")

        print("Servidor de Ollama listo.")

        break

    except requests.exceptions.ConnectionError:

        time.sleep(1)

else:

    print("El servidor no respondió. Vuelve a ejecutar esta celda.")

!ollama --version

Servidor de Ollama listo.
ollama version is 0.40.3


### **DESCARGAR LOS MODELOS**

Vamos a trabajar con dos tamaños de Llama: **Llama 3.2 de 3B** (ligero) y **Llama 3.1 de 8B** (más grande). Guardamos los nombres en variables para usar *exactamente la misma etiqueta* al descargar y al llamar al modelo; si no coinciden carácter por carácter, Ollama devuelve error.

In [ ]:
# Descargar los dos modelos Llama (la primera vez tarda unos minutos)
MODELO_LIGERO = "llama3.2:3b"

MODELO_GRANDE = "llama3.1:8b"

!ollama pull {MODELO_LIGERO}

!ollama pull {MODELO_GRANDE}

!ollama list



NAME           ID              SIZE      MODIFIED               
llama3.1:8b    46e0c10c039e    4.9 GB    Less than a second ago    
llama3.2:3b    a80c4f17acd5    2.0 GB    1 second ago              


### **CLIENTE DE PYTHON**

In [ ]:
# Instalar el cliente de Ollama y conectarlo al servidor local
!pip install ollama -q

import ollama

import time

client = ollama.Client(host="http://localhost:11434")

print("Cliente de Ollama inicializado correctamente.")

Cliente de Ollama inicializado correctamente.


## **DE PALABRAS A TOKENS**

### **PRIMERA LLAMADA A LLAMA (ZERO-SHOT)**

Un LLM predice el siguiente token más probable, **no busca la respuesta en una base de datos**. Antes de que Llama genere una sola palabra, tu prompt se divide en **tokens**.

Vamos a enviar el prompt a Llama en modo zero-shot (sin ejemplos previos) y ver la respuesta generada.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
prompt = "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?"

response = client.chat(

    model=MODELO_LIGERO,

    messages=[{"role": "user", "content": prompt}]
)

print(prompt)
print(response.message.content)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
**Diferencia entre la RAM y el almacenamiento**

La RAM (Random Access Memory) y el almacenamiento son dos conceptos fundamentales en la computadora, pero a menudo se confunden o utilizados de manera intercambiable. En este artículo, te explicaremos las diferencias clave entre los dos.

**RAM (Random Access Memory)**

La RAM es un tipo de memoria de acceso aleatorio que almacena datos y programas temporales. Su función principal es proporcionar acceso rápido y directo a los datos y programas que se están utilizando en la computadora. La RAM es voluble, lo que significa que los datos se pierden cuando la computadora se apaga.

**Almacenamiento**

El almacenamiento, por otro lado, se refiere a la capacidad de almacenar datos y programas permanentemente, incluso cuando la computadora se apaga. El almacenamiento puede ser de varios tipos, como:

* **Disco duro** (HD): un dispositivo de almacenamiento que utiliza di

In [ ]:
print(response.model_dump_json(indent=2))

{
  "model": "llama3.2:3b",
  "created_at": "2026-10-13T01:24:14.065164096Z",
  "done": true,
  "done_reason": "stop",
  "total_duration": 99472307388,
  "load_duration": 53921114777,
  "prompt_eval_count": 43,
  "prompt_eval_duration": 38896006000,
  "eval_count": 540,
  "eval_duration": 6651567000,
  "message": {
    "role": "assistant",
    "content": "**Diferencia entre la RAM y el almacenamiento**\n\nLa RAM (Random Access Memory) y el almacenamiento son dos conceptos fundamentales en la computadora, pero a menudo se confunden o utilizados de manera intercambiable. En este artículo, te explicaremos las diferencias clave entre los dos.\n\n**RAM (Random Access Memory)**\n\nLa RAM es un tipo de memoria de acceso aleatorio que almacena datos y programas temporales. Su función principal es proporcionar acceso rápido y directo a los datos y programas que se están utilizando en la computadora. La RAM es voluble, lo que significa que los datos se pierden cuando la computadora se apaga.\n\n

### **CUÁNTOS TOKENS CONSUMIÓ EL PROMPT**

La respuesta de Ollama trae el conteo de tokens de entrada (`prompt_eval_count`) y de salida (`eval_count`). Es la forma más directa de responder: ¿cuántos tokens consumió mi prompt?

In [ ]:
# Mostrar cuántos tokens tuvo el prompt y cuántos tuvo la respuesta
tokens_prompt = response.prompt_eval_count

tokens_respuesta = response.eval_count

print("Tokens del prompt:", tokens_prompt)

print("Tokens de la respuesta:", tokens_respuesta)

print("Tokens totales:", tokens_prompt + tokens_respuesta)

# Con Ollama no se factura: los tokens miden cuánto trabajo hizo el modelo y cuánto de su contexto usaste

Tokens del prompt: 43
Tokens de la respuesta: 540
Tokens totales: 583


### **MIDIENDO LA LATENCIA DE INFERENCIA**

Elegir el tamaño correcto de modelo es una decisión de ingeniería, no solo de potencia bruta. Vamos a medir cuánto tarda Llama en responder el mismo prompt.

In [ ]:
# Medir el tiempo de respuesta de Llama para el mismo prompt
inicio = time.time()

response_tiempo = client.chat(

    model=MODELO_LIGERO,

    messages=[{"role": "user", "content": prompt}]

)

duracion = time.time() - inicio

# duracion_ms = duracion * 1000  # si prefieres reportarlo en milisegundos

print(f"Tiempo de respuesta: {duracion:.2f} segundos")

Tiempo de respuesta: 9.58 segundos


### **COMPARANDO DOS TAMAÑOS DE MODELO**

Repite la misma llamada usando la versión más grande de Llama que cabe en la GPU gratuita de Colab (8B) y compara tiempo de respuesta y calidad contra el modelo ligero (3B).

Ojo: la primera vez que se usa un modelo, Ollama tiene que **cargarlo en la memoria de la GPU**. Ese tiempo de carga viene en `load_duration` y conviene separarlo para que la comparación sea justa.

In [ ]:
# Definir y enviar un prompt a Llama en Ollama y mostrar la respuesta generada
inicio = time.time()

response_grande = client.chat(

    model= MODELO_GRANDE,

    messages=[{"role": "user", "content": prompt}]

)

duracion_grande = time.time() - inicio

carga_grande = response_grande.load_duration / 1e9  # nanosegundos -> segundos

tokens_ligero = response.prompt_eval_count + response.eval_count

tokens_grande = response_grande.prompt_eval_count + response_grande.eval_count

print(f"Modelo ligero ({MODELO_LIGERO}): {duracion:.2f} s — {tokens_ligero} tokens")

print(f"Modelo grande ({MODELO_GRANDE}): {duracion_grande:.2f} s — {tokens_grande} tokens (de los cuales {carga_grande:.2f} s fueron carga del modelo)")

print("\nRespuesta del modelo grande:\n", response_grande.message.content)


Modelo ligero (llama3.2:3b): 9.58 s — 583 tokens
Modelo grande (llama3.1:8b): 19.79 s — 72 tokens (de los cuales 18.36 s fueron carga del modelo)

Respuesta del modelo grande:
 La RAM (Memoria de Acceso Aleatorio) se utiliza para ejecutar las aplicaciones y programas en tiempo real. Mientras que el almacenamiento se utiliza para almacenar los datos de forma permanente.


## **CHALLENGE: COMPARADOR DE MODELOS LLAMA**

Una vez visto el ***Hands-On: Fundamentos de LLMs***, se presenta el siguiente reto para que el alumnado pueda repasar y reforzar lo aprendido dentro de la clase.

Se construirá un pequeño **comparador que envíe 3 preguntas** reales a Llama y registre, para cada una, el **tiempo de respuesta** y el **número de tokens** consumidos.

**IMPORTANTE:** Para su revisión, **es indispensable que los apartados anteriores se encuentren llenados con el código visto durante la sesión.**

### **INSTRUCCIONES:**

**1. Carga la API key y genera 3 preguntas:**

   * Lee la API key de llama ya configurada desde **Colab Secrets**.
   * Construye una lista vacía llamada `preguntas` y agrégale 3 preguntas frecuentes de tu propio contexto (soporte técnico, tienda, escuela, etc.).

In [ ]:
# Leer API key desde Colab Secrets
from google.colab import userdata

api_key = userdata.get("LLAMA_API_KEY")

print("API key cargada correctamente." if api_key else "No se encontró la API key.")

API key cargada correctamente.


In [ ]:
# Definir la lista de preguntas
preguntas = [
    "¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?",
    "¿Cómo puedo solucionar un problema de conexión a Internet?",
    "¿Qué puedo hacer si mi computadora funciona lentamente?"
]

for prompt in preguntas:

    response = client.chat(

        model=MODELO_LIGERO,

        messages=[{"role": "user", "content": prompt}]

    )

    print(prompt)
    print(response.message.content)
    print("-" * 60)

¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?
La RAM (Random Access Memory) y el almacenamiento de una computadora son dos conceptos fundamentalmente diferentes.

**RAM (Memoria de Acceso Aleatorio)**

La RAM es una memoria temporal que almacena los datos y los programas en ejecución mientras se procesan. Es una memoria volátil, lo que significa que se pierde su contenido cuando se apaga el poder. La RAM es utilizada por el procesador para acceder a los datos y realizar cálculos.

**Almacenamiento**

El almacenamiento, por otro lado, se refiere al espacio donde se almacenan los datos y los programas que no se utilizan actualmente. Los tipos de almacenamiento más comunes son:

*   Disco duro (HD)
*   Disco sólido (SSD)
*   Tarjeta de memoria (RAM)
*   Disco externo

El almacenamiento es una memoria no volátil, lo que significa que se conserva su contenido incluso cuando se apaga el poder.

**Diferencias clave**

A continuación, se presentan las diferencias 

**2. Consulta la primera pregunta:** Envía la primera pregunta a Llama usando el modelo más ligero disponible en Groq. Guarda la respuesta, el tiempo de respuesta y el número de tokens en un diccionario llamado `resultado_1`.

In [ ]:
# Consultar la primera pregunta y guardar el resultado en resultado_1
import time

# Corregimos el modelo ligero para que apunte al descargado en Ollama
MODELO_LIGERO = "llama3.2:3b"

inicio = time.perf_counter()

respuesta = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[0]}]
)

tiempo = time.perf_counter() - inicio

resultado_1 = {
    "pregunta": preguntas[0],
    "respuesta": respuesta.message.content,
    "tiempo_respuesta": tiempo,
    "tokens": respuesta.eval_count + respuesta.prompt_eval_count
}

print(resultado_1)

{'pregunta': '¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?', 'respuesta': '**Diferencia entre RAM y almacenamiento en una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales de una computadora que funcionan de manera diferente.\n\n**RAM (Random Access Memory)**\n\nLa RAM es una parte del sistema que almacena datos y programas en la memoria principal de la computadora. Es un tipo de memoria temporal que permite que la computadora acceda rápidamente a los datos que necesita procesar. La RAM es como una "cámara de trabajo" donde la computadora puede acceder a los datos y realizar operaciones sin tener que buscarlos en el disco duro.\n\n**Características de la RAM:**\n\n* Es volátil, lo que significa que se pierden cuando se apaga la computadora.\n* Es rápida, lo que permite que la computadora realice operaciones en tiempo real.\n* Es limitada en capacidad, lo que significa que hay un límite en la cantidad de d

**3. Repite para la segunda y tercera pregunta:** Crea `resultado_2` y `resultado_3` de la misma forma.

In [ ]:
# Consultar la segunda pregunta y guardar el resultado en resultado_2
inicio = time.perf_counter()

respuesta = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[1]}]
)

tiempo = time.perf_counter() - inicio

resultado_2 = {
    "pregunta": preguntas[1],
    "respuesta": respuesta.message.content,
    "tiempo_respuesta": tiempo,
    "tokens": respuesta.eval_count + respuesta.prompt_eval_count
}

print(resultado_2)

{'pregunta': '¿Cómo puedo solucionar un problema de conexión a Internet?', 'respuesta': 'Solucionar un problema de conexión a Internet puede ser un proceso desafiante, pero hay varias pasos que puedes seguir para intentar resolver el problema. Aquí te presento algunos consejos y pasos que puedes seguir:\n\n**1. Verifica el estado de tu equipo**\n\n* Asegúrate de que tu equipo esté encendido y que la fuente de alimentación esté funcionando correctamente.\n* Verifica que el conector de Internet esté correctamente conectado y que no esté dañado.\n\n**2. Verifica la configuración de tu equipo**\n\n* Asegúrate de que la configuración de tu equipo para Internet esté correcta.\n* Verifica que el número de serie de tu equipo esté configurado correctamente en tu router.\n* Asegúrate de que el protocolo de Internet esté configurado correctamente (por ejemplo, DHCP o static).\n\n**3. Verifica la red**\n\n* Verifica que la red esté activa y que no haya problemas con la red de tu hogar.\n* Asegúrat

In [ ]:
# Consultar la tercera pregunta y guardar el resultado en resultado_3
inicio = time.perf_counter()

respuesta = client.chat(
    model=MODELO_LIGERO,
    messages=[{"role": "user", "content": preguntas[2]}]
)

tiempo = time.perf_counter() - inicio

resultado_3 = {
    "pregunta": preguntas[2],
    "respuesta": respuesta.message.content,
    "tiempo_respuesta": tiempo,
    "tokens": respuesta.eval_count + respuesta.prompt_eval_count
}

print(resultado_3)

{'pregunta': '¿Qué puedo hacer si mi computadora funciona lentamente?', 'respuesta': '**Soluciones para mejorar el rendimiento de tu computadora**\n\nSi tu computadora funciona lentamente, hay varias cosas que puedes hacer para mejorar su rendimiento. Aquí te presento algunas soluciones:\n\n### 1. **Reinicia tu computadora**\n\nA veces, una simple reinicia puede resolver muchos problemas. Cierra todas las aplicaciones y programas que estén abiertos, y luego reinicia tu computadora.\n\n### 2. **Verifica los recursos del sistema**\n\n*   Revisa los recursos del sistema, como la cantidad de memoria RAM, el disco duro disponible y el procesador. Asegúrate de que estén dentro de los límites recomendados para tu sistema operativo.\n*   Puedes hacer esto desde el menú de inicio o desde el panel de control (en Windows) o desde la Terminal (en macOS).\n\n### 3. **Desinstala programas innecesarios**\n\n*   Revisa las aplicaciones que estás ejecutando y desinstala aquellas que no utilizas o que n

**4. Junta los resultados:** Agrega `resultado_1`, `resultado_2` y `resultado_3` a una lista vacía llamada `resultados`.

In [ ]:
# Definir la lista resultados y agregar los tres diccionarios
for i, resultado in enumerate(resultados, start=1):
    print(f"\n{'=' * 60}")
    print(f"              RESULTADO {i}")
    print("=" * 60)

    print(f"\nPregunta:\n{resultado['pregunta']}")
    print(f"\nRespuesta:\n{resultado['respuesta']}")
    print(f"\nTiempo de respuesta: {resultado['tiempo_respuesta']:.2f} segundos")
    print(f"Tokens utilizados: {resultado['tokens']}")

    print("\n")



              RESULTADO 1

Pregunta:
¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?

Respuesta:
**Diferencia entre RAM y almacenamiento en una computadora**

La RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales de una computadora que funcionan de manera diferente.

**RAM (Random Access Memory)**

La RAM es una parte del sistema que almacena datos y programas en la memoria principal de la computadora. Es un tipo de memoria temporal que permite que la computadora acceda rápidamente a los datos que necesita procesar. La RAM es como una "cámara de trabajo" donde la computadora puede acceder a los datos y realizar operaciones sin tener que buscarlos en el disco duro.

**Características de la RAM:**

* Es volátil, lo que significa que se pierden cuando se apaga la computadora.
* Es rápida, lo que permite que la computadora realice operaciones en tiempo real.
* Es limitada en capacidad, lo que significa que hay un límite en la canti

**5. Muestra la comparación:** Imprime `resultados` y concluye si el modelo ligero resolvió las 3 preguntas satisfactoriamente.

In [ ]:
# Mostrar la tabla final de resultados
print(resultados)

# Comparacion de las respuestas
for i, r in enumerate(resultados, 1):
    print(f"\nPregunta {i}: {r['pregunta']}")
    print(f"Respuesta: {r['respuesta']}")
    print(f"Tiempo: {r['tiempo_respuesta']:.2f} segundos")
    print(f"Tokens: {r['tokens']}")
    print("-" * 30)

# Conclusion
print("\nConclusion:")
print("El modelo ligero respondio las tres preguntas. "
      "Las respuestas se deben revisar para comprobar que sean correctas y utiles.")


[{'pregunta': '¿Cuál es la diferencia entre la RAM y el almacenamiento en una computadora?', 'respuesta': '**Diferencia entre RAM y almacenamiento en una computadora**\n\nLa RAM (Random Access Memory) y el almacenamiento son dos componentes fundamentales de una computadora que funcionan de manera diferente.\n\n**RAM (Random Access Memory)**\n\nLa RAM es una parte del sistema que almacena datos y programas en la memoria principal de la computadora. Es un tipo de memoria temporal que permite que la computadora acceda rápidamente a los datos que necesita procesar. La RAM es como una "cámara de trabajo" donde la computadora puede acceder a los datos y realizar operaciones sin tener que buscarlos en el disco duro.\n\n**Características de la RAM:**\n\n* Es volátil, lo que significa que se pierden cuando se apaga la computadora.\n* Es rápida, lo que permite que la computadora realice operaciones en tiempo real.\n* Es limitada en capacidad, lo que significa que hay un límite en la cantidad de 